In [17]:
import os
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

In [2]:
# Load a free embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [7]:
# Example
text = "Where can I find housing support in cote de neige?"
embedding = model.encode(text)
print(embedding)  # embedding vector
print(len(embedding))  # vector dimension

[-2.19896007e-02 -3.18913546e-04  5.33407629e-02  1.35475984e-02
 -3.16205584e-02 -2.06695348e-02 -6.40707314e-02  1.78082921e-02
  1.30646843e-02 -1.63816679e-02  7.29172230e-02 -1.53596744e-01
  4.18300517e-02 -5.09774610e-02  1.84632689e-02 -1.92130599e-02
  2.99762785e-02  2.26118136e-02  6.27642199e-02 -8.21513087e-02
 -5.60715795e-02  2.36295350e-02 -1.69564616e-02 -9.09102187e-02
  7.71306595e-03 -7.60629550e-02 -1.48924757e-02 -4.38313931e-02
  1.35318823e-02 -1.24977678e-02  1.25952765e-01 -4.76399697e-02
 -1.28472885e-02 -2.41492856e-02  8.17046762e-02  2.08590012e-02
  3.85004394e-02 -4.57497034e-03 -2.31864378e-02  1.47311632e-02
 -7.89561588e-03  3.72229214e-03 -8.32301157e-05 -6.82664961e-02
  3.63726704e-03  5.81119694e-02 -2.53730956e-02 -3.58305946e-02
  5.80512509e-02 -1.78331621e-02  4.56432067e-03  6.73287138e-02
  3.01482826e-02  9.59330946e-02  1.64239090e-02  2.48772018e-02
  2.98072305e-02  3.93492207e-02  2.55820137e-02 -2.46457681e-02
  1.51225150e-01  5.64317

In [13]:
print("Répertoire courant:", os.getcwd()) # get current working direcotry

Répertoire courant: d:\mscudemproject\main\dynamic_rags\notebooks


In [ ]:
# load chunks
with open("../data/processed/chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

In [16]:
embeddings = []
for chunk in chunks:
    chunk_emb = model.encode(chunk["text"])
    embeddings.append(chunk_emb)

embeddings = np.array(embeddings, dtype='float32')


In [19]:
len(embeddings)  # number of chunks

6816

In [ ]:
dim = embeddings.shape[1]  # e.g., 384
index = faiss.IndexFlatL2(dim)
index.add(embeddings)
print(f"FAISS index contains {index.ntotal} vectors")


FAISS index contains 6816 vectors


In [20]:
query = "I need help finding a daycare for my child"
query_emb = model.encode(query).astype('float32')

k = 5  # number of similar chunks to retrieve
D, I = index.search(np.array([query_emb]), k)
for i in I[0]:
    print(chunks[i]["text"])


Hours: Monday to Friday 9h00 to 17h00 Fees: drop-in daycare: $7 per day or $5 per half-day Financing: provincial, municipal Legal status: non-profit organization RELANCE JEUNES ET FAMILLES (LA) 2260 Ontario Street, Ville-Marie, Montréal, QC, H2K 1V8 514 525-1508 ext. 243 Website: www.relance.org Email: info@relance.org Services: Educational community organization.  Childhood and family * Daycare for 0 to 5-year-olds. * Stimulation and motor skills activities for 1 to 5-year-olds. * Program for 0 to 5-year-olds and 6 to 12-year-olds. * Youth clubs for 4 to 5-year-olds and 6 to 12-year-olds. * Program integrated into the primary schools of the neighbourhood. * School support. * Family follow-up. * Global child development. * Public education workshops. Support for parents * Accompaniment and psychosocial follow-up. * Family activities and outings. * Y'APP: parenting skills program. * Breastfeeding drop-in.  Food * Community kitchen. Education and Literacy 789
education workshops. * Perso

In [21]:
# category_filter = "Child, Youth and Family"
# filtered_indices = [i for i, c in enumerate(chunks) if c["category_name"] == category_filter]

# filtered_embeddings = embeddings[filtered_indices]
# filtered_index = faiss.IndexFlatL2(dim)
# filtered_index.add(filtered_embeddings)


In [2]:
import json
import faiss
from sentence_transformers import SentenceTransformer
import numpy as np

# Charger les chunks
with open("chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

texts = [c["text"] for c in chunks]

# Encoder les textes en vecteurs
embedder = SentenceTransformer("all-MiniLM-L6-v2")  # modèle léger
embeddings = embedder.encode(texts, convert_to_numpy=True)

# Créer l'index FAISS
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"Index créé avec {index.ntotal} vecteurs")


PackageNotFoundError: The 'numpy>=1.17' distribution was not found and is required by this application. 
Try: `pip install transformers -U` or `pip install -e '.[dev]'` if you're working with git main